# Práctica: Transferencia de Aprendizaje (Transfer Learning) y Ajuste Fino (Fine-Tuning)
## Reutilización de la Red VGG16 sobre el Dataset CIFAR-10

**Cátedra:** Visión por Computadora —
**FICH-UNL (Santa Fe, Argentina)**

### **Objetivos del Trabajo Práctico**
1. **Comprender la Extracción de Características (*Feature Extraction*):** Reutilizar una arquitectura convolucional profunda (VGG16) pre-entrenada sobre ImageNet congelando sus pesos para aprovechar sus jerarquías de filtros de bajo y medio nivel.
2. **Implementar Transferencia de Aprendizaje (*Transfer Learning*):** Adaptar la cabeza clasificadora (*MLP Top*) reemplazando la salida original de 1,000 clases por una adecuada para las 10 clases de CIFAR-10.
3. **Aplicar Ajuste Fino (*Fine-Tuning*):** Descongelar selectivamente las capas convolucionales superiores de VGG16 y reentrenar con una tasa de aprendizaje reducida (\(\alpha = 10^{-4}\)) para ajustar los campos receptivos de alto nivel al nuevo dominio.
4. **Evaluar y diagnosticar el rendimiento:** Interpretar el desempeño del modelo mediante curvas de entrenamiento, inferencia sobre muestras aleatorias y el análisis de la Matriz de Confusión.

---

## 📖 **Marco Teórico y Fundamentos Metodológicos**

Entrenar arquitecturas convolucionales profundas desde cero requiere enormes volúmenes de datos etiquetados y un costo computacional elevado. La **Transferencia de Aprendizaje (*Transfer Learning*)** resuelve este desafío permitiendo adaptar conocimientos aprendidos en un problema fuente (ej. ImageNet con 1.2M de imágenes) hacia un problema destino (ej. CIFAR-10).

```
  ┌────────────────────────────────────────────────────────────────────────┐
  │                     ARQUITECTURA DE TRANSFERENCIA                      │
  └────────────────────────────────────────────────────────────────────────┘
   [ Imagen 32x32x3 ] ──► [ Bloques Convolucionales VGG16 ] ──► [ GlobalAvgPool ] ──► [ Dense(1024) ] ──► [ Softmax(10) ]
                          └─────────────┬───────────────┘   └──────────────────────┬──────────────────────┘
                                        │                                          │
                          Etapa 1: Pesos Congelados                   Cabeza Clasificadora Entrenable
                          Etapa 2: Últimos 4 Bloques Descongelados
```

### 1. Extracción de Características (*Feature Extraction*)
Las primeras capas de una CNN detectan bordes, esquinas y texturas genéricas útiles para cualquier tarea visual. Al remover la capa de clasificación original (`include_top=False`), utilizamos la base convolucional como una extractora de características fija, entrenando únicamente la nueva cabeza clasificadora.

### 2. Ajuste Fino (*Fine-Tuning*)
Una vez que las capas densas clasificadoras han sido pre-entrenadas y sus pesos inicializados correctamente, descongelamos las capas convolucionales superiores. Al reentrenar estas capas con una **tasa de aprendizaje muy pequeña** (por ejemplo, \(10^{-4}\)), permitimos que los filtros de alto nivel se adapten a las particularidades del nuevo dataset sin destruir el conocimiento previo (*catastrophic forgetting*).

---

### 📚 **Lecturas Sugeridas (Conceptuales y Divulgativas)**

- [A Gentle Introduction to Transfer Learning for Deep Learning (Machine Learning Mastery)](https://machinelearningmastery.com/transfer-learning-for-deep-learning/)
  *Una explicación clara y accesible sobre qué es la transferencia de aprendizaje, por qué reutilizamos modelos pre-entrenados y la diferencia entre usar un modelo como extractor o hacer fine-tuning.*

- [Transfer Learning and Fine-Tuning Tutorial (TensorFlow Guide)](https://www.tensorflow.org/tutorials/images/transfer_learning)
  *Guía conceptual sobre cómo congelar capas, entrenar capas superiores y ajustar paulatinamente los bloques convolucionales en Keras.*

- [Understanding the Confusion Matrix in Machine Learning (Towards Data Science)](https://towardsdatascience.com/understanding-confusion-matrix-a9feb424f007)
  *Un artículo breve y gráfico que enseña cómo interpretar matrices de confusión y evaluar los errores entre clases visualmente similares.*

---

## Carga de Librerías

In [ ]:
# Importar librerías necesarias
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.utils import to_categorical

## **Ejercicio 1: Carga, Exploración y Preprocesamiento del Dataset CIFAR-10**

El dataset **CIFAR-10** consiste en 60,000 imágenes en color de $32 \times 32$ píxeles distribuidas en 10 clases (50,000 para entrenamiento y 10,000 para validación/prueba).

> **Observación**: para no descargar todo el dataset y para que las pruebas sean más rápidas, sólo descargamos los datos de **test** y lo dividimos en los sub-conjuntos.


En este ejercicio:
1. Cargarás el dataset e inspeccionarás visualmente algunas muestras etiquetadas.
2. Normalizarás los valores de intensidad al rango $[0.0, 1.0]$.
3. Aplicarás codificación *One-Hot Encoding* sobre las etiquetas numéricas $y \in \{0, 1, \dots, 9\}$.

In [ ]:
!pip install -q datasets scikit-learn

import random
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Mapeo numérico a nombres reales de las clases de CIFAR-10
class_names = ['Avión', 'Automóvil', 'Pájaro', 'Gato', 'Ciervo',
               'Perro', 'Rana', 'Caballo', 'Barco', 'Camión']

# Descarga SOLO el split de test de CIFAR-10 (10.000 imágenes)
ds = load_dataset("uoft-cs/cifar10", split="test")

X_all = np.stack([np.array(img) for img in ds["img"]])   # (10000, 32, 32, 3) uint8
y_all = np.array(ds["label"])                            # (10000,)

# 9.000 train / 1.000 validación, estratificado
X_train, X_val, y_train_raw, y_val_raw = train_test_split(
    X_all, y_all, test_size=1000, stratify=y_all, random_state=42
)

# Mismo formato que Keras: etiquetas con forma (N, 1)
y_train_raw = y_train_raw.reshape(-1, 1)
y_val_raw   = y_val_raw.reshape(-1, 1)

print(f"Dimensiones de Entrenamiento: {X_train.shape}")
print(f"Dimensiones de Validación: {X_val.shape}")
print("Ejemplos por clase (train):", np.bincount(y_train_raw.ravel()))
print("Ejemplos por clase (val):  ", np.bincount(y_val_raw.ravel()))

# Visualizar 10 ejemplos al azar del conjunto de entrenamiento
plt.figure(figsize=(12, 5))
random_indices = random.sample(range(len(X_train)), 10)

for i, idx in enumerate(random_indices):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train[idx])
    label_idx = y_train_raw[idx][0]
    plt.title(f"{class_names[label_idx]} (#{label_idx})")
    plt.axis('off')
plt.tight_layout()
plt.show()

# Normalizar píxeles al intervalo [0, 1]
X_train = X_train.astype('float32') / 255.0
X_val = X_val.astype('float32') / 255.0

# Codificación One-Hot de las etiquetas
y_train = to_categorical(y_train_raw, num_classes=10)
y_val = to_categorical(y_val_raw, num_classes=10)

print(f"Forma de las etiquetas transformadas (One-Hot): {y_train.shape}")

### 💡 Reflexión - Ejercicio 1: Discrepancia de Resolución y Representación

1. **Efecto de la Resolución Espacial:**
  * La red **VGG16** fue diseñada originalmente para procesar imágenes de $224 \times 224 \times 3$ píxeles en ImageNet. Sin embargo, en CIFAR-10 le entregamos imágenes de $32 \times 32 \times 3$.
  * Explica qué ocurre con la dimensión del mapa de características tras pasar por las 5 etapas de *Max Pooling* de VGG16 (donde la dimensión espacial se reduce a la mitad en cada bloque: $32 \rightarrow 16 \rightarrow 8 \rightarrow 4 \rightarrow 2 \rightarrow 1$).

2. **Capas Globales vs Flatten:**
  * ¿Qué función cumple la capa `GlobalAveragePooling2D()` al conectarse al final de la base convolucional y por qué es superior a aplanar el tensor mediante `Flatten()` en términos de cantidad de parámetros entrenables?

---

## **Ejercicio 2: Transferencia de Aprendizaje (Transfer Learning) con VGG16**

En esta etapa instanciaremos el extractor de características **VGG16** pre-entrenado en ImageNet sin su MLP superior (`include_top=False`).

Pasos a realizar:
1. Congelar los pesos de la base convolucional (`layer.trainable = False`).
2. Agregar una nueva cabeza clasificadora compuestas por `GlobalAveragePooling2D`, una capa `Dense` de 1024 neuronas con activación ReLU y una capa `Dense` final de 10 neuronas con activación `softmax`.
3. Entrenar el modelo durante 2 épocas utilizando el optimizador `Adam` y la función de pérdida `categorical_crossentropy`.

In [ ]:
# Cargar la base VGG16 sin las capas densas superiores
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Construir la nueva cabeza clasificadora
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
predictions = Dense(10, activation='softmax')(x)

pretrain_model = Model(inputs=base_model.input, outputs=predictions)

# Congelar todas las capas de la base VGG16
for layer in base_model.layers:
    layer.trainable = False

# Compilar el modelo para Transfer Learning
pretrain_model.compile(optimizer='adam',
                       loss='categorical_crossentropy',
                       metrics=['accuracy'])

pretrain_model.summary()

In [ ]:
# Entrenar únicamente la cabeza clasificadora (Transfer Learning)
print("=== Entrenando Cabeza Clasificadora (Base Congelada) ===")
history_pretrain = pretrain_model.fit(
    X_train, y_train,
    epochs=2,
    batch_size=64,
    validation_data=(X_val, y_val)
)

### 💡 Reflexión - Ejercicio 2: Extracción de Características y Congelamiento

1. **Parámetros Entrenables vs. No Entrenables:**
  * Observa el resumen (`summary()`) generado por Keras. ¿Cuántos parámetros totales tiene el modelo y cuántos son efectivamente entrenables durante esta primera fase?
  * Justifica por qué es crítico mantener la base congelada mientras se inicializan los pesos de la cabeza clasificadora aleatoria.
  * Si quieren pueden evaluar distintos valores para batch y cantidad de épocas, mientras verifican sus métricas

---

## **Ejercicio 3: Ajuste Fino (Fine-Tuning) de Bloques Convolucionales Profundos**

Una vez adaptada la cabeza clasificadora, procederemos a realizar **Fine-Tuning** descongelando las últimas 4 capas convolucionales de VGG16.

Pasos a realizar:
1. Descongelar las últimas 4 capas de `base_model` (`layer.trainable = True`).
2. Re-compilar el modelo reduciendo drásticamente la tasa de aprendizaje a \(\alpha = 10^{-4}\) (`learning_rate=0.0001`).
3. Reentrenar el modelo durante 2 épocas adicionales y evaluar el impacto en la precisión.

In [ ]:
# Descongelar las últimas 4 capas convolucionales de VGG16
for layer in base_model.layers[-4:]:
    layer.trainable = True

# Re-compilar con una tasa de aprendizaje reducida para Fine-Tuning
pretrain_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

pretrain_model.summary()

In [ ]:
# Entrenar en la fase de Fine-Tuning
print("=== Ejecutando Fine-Tuning (Últimas 4 Capas Descongeladas) ===")
history_finetune = pretrain_model.fit(
    X_train, y_train,
    epochs=2,
    batch_size=64,
    validation_data=(X_val, y_val)
)

### 💡 Reflexión - Ejercicio 3: Estrategia de Fine-Tuning y Tasa de Aprendizaje

1. **Selección de la Tasa de Aprendizaje (Learning Rate):**
  * ¿Por qué es fundamental reducir la tasa de aprendizaje a \(10^{-4}\) durante el Fine-Tuning en lugar de usar la tasa por defecto (\(10^{-3}\)) de Adam?
  * ¿Qué sucedería con las representaciones pre-entrenadas de ImageNet si utilizáramos una tasa de aprendizaje muy grande al ajustar los bloques convolucionales?

2. **Nivel de Abstracción en las Capas:**
  * Explica por qué se eligen descongelar las **últimas capas** (las más cercanas a la salida) en lugar de las primeras capas convolucionales de la red.

---

## **Ejercicio 4: Inferencia Individual, Matriz de Confusión y Diagnóstico**

En este último ejercicio evaluaremos cuantitativa y cualitativamente el desempeño final del modelo ajustado sobre el conjunto de validación:

1. Inspeccionaremos de forma visual la predicción del modelo sobre 5 muestras aleatorias de validación.
2. Generaremos y graficaremos la **Matriz de Confusión** etiquetada para identificar qué clases confunde la red con mayor frecuencia.
3. Verificaremos el balance de imágenes por clase en el conjunto de validación.

In [ ]:
# Probar la clasificación en 5 imágenes al azar del conjunto de validación
num_test_samples = 5
random_val_indices = random.sample(range(len(X_val)), num_test_samples)

plt.figure(figsize=(15, 3))
for i, idx in enumerate(random_val_indices):
    test_img = X_val[idx]
    true_class_idx = np.argmax(y_val[idx])

    # Inferencia (agregando dimensión de lote)
    input_tensor = tf.expand_dims(test_img, axis=0)
    prediction = pretrain_model.predict(input_tensor, verbose=0)
    pred_class_idx = np.argmax(prediction, axis=1)[0]
    confidence = prediction[0][pred_class_idx] * 100

    plt.subplot(1, 5, i + 1)
    plt.imshow(test_img)
    color = 'green' if true_class_idx == pred_class_idx else 'red'
    title = f"Real: {class_names[true_class_idx]}\nPred: {class_names[pred_class_idx]}\n({confidence:.1f}%)"
    plt.title(title, color=color, fontsize=11)
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Predecir etiquetas para todo el conjunto de validación
print("Calculando predicciones en el dataset de validación completo...")
y_pred_probs = pretrain_model.predict(X_val, batch_size=64, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_val, axis=1)

# Generar Matriz de Confusión
cm = confusion_matrix(y_true, y_pred)

# Visualizar la Matriz de Confusión mediante Seaborn
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Etiqueta Predicha', fontsize=12)
plt.ylabel('Etiqueta Real', fontsize=12)
plt.title('Matriz de Confusión en Validación (CIFAR-10 + VGG16 Fine-Tuned)', fontsize=14)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.show()

In [ ]:
# Verificar la distribución y número de imágenes por clase en el set de validación
class_counts = Counter(y_true)

print("=== Distribución de Muestras en el Set de Validación ===")
for class_idx, count in sorted(class_counts.items()):
    print(f"Clase {class_idx:2d} ({class_names[class_idx]:10s}): {count} imágenes")

### 💡 Reflexión Guía - Ejercicio 4: Análisis de Confusiones y Errores de Dominio

1. **Análisis de la Matriz de Confusión:**
  * Observa los valores fuera de la diagonal principal en la matriz de confusión. ¿Entre qué pares de clases se producen las mayores confusiones (por ejemplo, *Gato* vs. *Perro* o *Automóvil* vs. *Camión*)?
  * Explica qué similitudes visuales o semánticas justifican que la red confunda esas categorías.

2. **Métricas y Balance de Clases:**
  * ¿Está balanceado el dataset de validación de CIFAR-10 según la distribución obtenida? ¿Por qué esta condición facilita la interpretación de la Precisión (*Accuracy*) global sin caer en sesgos por clases dominantes?

---